## Author: Gabriel Vance
## Class: DSC550 - Data Mining
## Professor: Dr. Brett Werner
## Date: June 17, 2025

# Purpose: This notebook performs various tasks labeled "Exercises" for Week 3 of the class. Data must be present within the folder that this notebook is run in.
# This week focuses on Text/Sentiment Analysis, Categorical Data, and Dates/Times

In [35]:
import pandas as pd
from textblob import TextBlob
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from nltk.sentiment import SentimentIntensityAnalyzer

## Import the movie review data as a data frame and ensure that the data is loaded properly.


In [36]:
# Read training data
df = pd.read_csv('training_data.tsv', sep='\t')

# View the first few rows
print(df.head(5))

       id  sentiment                                             review
0  5814_8          1  With all this stuff going down at the moment w...
1  2381_9          1  \The Classic War of the Worlds\" by Timothy Hi...
2  7759_3          0  The film starts with a manager (Nicholas Bell)...
3  3630_4          0  It must be assumed that those who praised this...
4  9495_8          1  Superbly trashy and wondrously unpretentious 8...


## How many of each positive and negative reviews are there?


In [37]:
# Count positive and negative reviews
counts = df['sentiment'].value_counts()

print("Negative reviews (0):", counts.get(0, 0))
print("Positive reviews (1):", counts.get(1, 0))

Negative reviews (0): 12500
Positive reviews (1): 12500


## Use TextBlob to classify each movie review as positive or negative. Assume that a polarity score greater than or equal to zero is a positive sentiment and less than 0 is a negative sentiment.


In [38]:
# Classify each review using TextBlob polarity
def get_textblob_sentiment(text):
    return 1 if TextBlob(str(text)).sentiment.polarity >= 0 else 0

df['textblob_sentiment'] = df['review'].apply(get_textblob_sentiment)
print(df[['sentiment', 'textblob_sentiment']].head(10))

   sentiment  textblob_sentiment
0          1                   1
1          1                   1
2          0                   0
3          0                   1
4          1                   0
5          1                   1
6          0                   0
7          0                   1
8          0                   1
9          1                   1


## Check the accuracy of this model. Is this model better than random guessing?


In [39]:
# Calculate accuracy of TextBlob sentiment predictions
accuracy = (df['sentiment'] == df['textblob_sentiment']).mean()
print(f"TextBlob sentiment accuracy: {accuracy:.4f}")

# Compare to random guessing (expected accuracy = 0.5 for balanced classes)
print("The TextBlob sentiment analysis accuracy is significantly better than random guessing (0.5).")

TextBlob sentiment accuracy: 0.6852
The TextBlob sentiment analysis accuracy is significantly better than random guessing (0.5).


## For up to five points extra credit, use another prebuilt text sentiment analyzer, e.g., VADER, and repeat steps (3) and (4).


In [40]:


# Download VADER
nltk.download('vader_lexicon', quiet=True)

# Initialize VADER
sia = SentimentIntensityAnalyzer()

# Classify each review using VADER compound score
def get_vader_sentiment(text):
    score = sia.polarity_scores(str(text))['compound']
    return 1 if score >= 0 else 0

df['vader_sentiment'] = df['review'].apply(get_vader_sentiment)


accuracy = (df['sentiment'] == df['vader_sentiment']).mean()
print(f"VADER sentiment accuracy: {accuracy:.4f}")
print("VADER took about 22s whereas TextBlob took 12s. VADER only had 1% more accuracy and was much slower.")

VADER sentiment accuracy: 0.6936
VADER took about 22s whereas TextBlob took 12s. VADER only had 1% more accuracy and was much slower.


## Convert all text to lowercase letters.


In [41]:
# Convert reviews to lowercase for consistency
df['review'] = df['review'].str.lower()
print(df['review'].head(5))

0    with all this stuff going down at the moment w...
1    \the classic war of the worlds\" by timothy hi...
2    the film starts with a manager (nicholas bell)...
3    it must be assumed that those who praised this...
4    superbly trashy and wondrously unpretentious 8...
Name: review, dtype: object


## Remove punctuation and special characters from the text.


In [42]:

# Remove punctuation and special characters from the review text
df['review'] = df['review'].apply(lambda x: re.sub(r'[^a-zA-Z0-9\s]', '', x))
print(df['review'].head(5))

0    with all this stuff going down at the moment w...
1    the classic war of the worlds by timothy hines...
2    the film starts with a manager nicholas bell g...
3    it must be assumed that those who praised this...
4    superbly trashy and wondrously unpretentious 8...
Name: review, dtype: object


## Remove stop words.


In [43]:

# Download stopwords if not already downloaded
nltk.download('stopwords', quiet=True)

stop_words = set(stopwords.words('english'))

# Remove stop words from the review text
df['review'] = df['review'].apply(lambda x: ' '.join([word for word in x.split() if word not in stop_words]))
print(df['review'].head(5))


0    stuff going moment mj ive started listening mu...
1    classic war worlds timothy hines entertaining ...
2    film starts manager nicholas bell giving welco...
3    must assumed praised film greatest filmed oper...
4    superbly trashy wondrously unpretentious 80s e...
Name: review, dtype: object


## Apply NLTK’s PorterStemmer.


In [44]:

# Initialize the PorterStemmer
stemmer = PorterStemmer()

# Apply stemming to each word in the review text
df['review'] = df['review'].apply(lambda x: ' '.join([stemmer.stem(word) for word in x.split()]))
print(df['review'].head(5))

0    stuff go moment mj ive start listen music watc...
1    classic war world timothi hine entertain film ...
2    film start manag nichola bell give welcom inve...
3    must assum prais film greatest film opera ever...
4    superbl trashi wondrous unpretenti 80 exploit ...
Name: review, dtype: object


## Create a bag-of-words matrix from your stemmed text (output from (4)), where each row is a word-count vector for a single movie review (see sections 5.3 & 6.8 in the Machine Learning with Python Cookbook). Display the dimensions of your bag-of-words matrix. The number of rows in this matrix should be the same as the number of rows in your original data frame.


In [50]:

# Create a CountVectorizer instance
vectorizer = CountVectorizer()

# Fit and transform the stemmed review text to create the bag-of-words matrix
bow_matrix = vectorizer.fit_transform(df['review'])

# Display the dimensions of the bag-of-words matrix
print("Bag-of-words matrix shape:", bow_matrix.shape)

# To check that the number of rows in this matrix matches the number of rows
print("Number of reviews:", df.shape)

Bag-of-words matrix shape: (25000, 92226)
Number of reviews: (25000, 5)


## Create a term frequency-inverse document frequency (tf-idf) matrix from your stemmed text, for your movie reviews (see section 6.9 in the Machine Learning with Python Cookbook). Display the dimensions of your tf-idf matrix. These dimensions should be the same as your bag-of-words matrix.


In [51]:

#! Good article for reference: https://www.capitalone.com/tech/machine-learning/understanding-tf-idf/
# Create a TfidfVectorizer instance
tfidf_vectorizer = TfidfVectorizer()

# Fit and transform the stemmed review text to create the tf-idf matrix
tfidf_matrix = tfidf_vectorizer.fit_transform(df['review'])

# Display the dimensions of the tf-idf matrix
print("TF-IDF matrix shape:", tfidf_matrix.shape)

TF-IDF matrix shape: (25000, 92226)
